# Fase 3 - Analise Final NDVI em CRISP-DM

Notebook da fase 3, com consolidacao da analise NDVI em estrutura CRISP-DM.

Estrutura:
- Business Understanding
- Data Understanding
- Data Preparation
- Modeling
- Evaluation
- Deliverables

Premissas da modelagem:
- `NDVI` continua sendo o eixo central.
- `clima`, `MIIP`, `solo` e `operacao` entram como variaveis explicadoras.
- a modelagem foi mantida interpretavel; nao foi usado deep learning porque a base atual e pequena e nao tem target final de colheita consolidado em todos os pares.


In [ ]:
        from __future__ import annotations

        import base64

import importlib.util
        import json
        import os
        import subprocess
        import sys
        from pathlib import Path


        OUTPUT_SUBDIR = 'notebook_outputs/phase3_ndvi'
        NOTEBOOK_MODE = os.environ.get("MONOLITHFARM_NOTEBOOK_MODE", "auto").strip().lower()
        if NOTEBOOK_MODE not in {"auto", "jupyter", "colab"}:
            raise ValueError("MONOLITHFARM_NOTEBOOK_MODE deve ser `auto`, `jupyter` ou `colab`.")

        PROFILE_NAME = os.environ.get("MONOLITHFARM_PROFILE", "").strip()
        CONFIG_ENV_PATH = os.environ.get("MONOLITHFARM_PATHS_FILE", "").strip()
        IN_COLAB_RUNTIME = "google.colab" in sys.modules
        USE_COLAB_MODE = NOTEBOOK_MODE == "colab" or (NOTEBOOK_MODE == "auto" and IN_COLAB_RUNTIME)
        REQUIRED_PACKAGES = ['duckdb', 'pandas', 'plotly', 'pyproj', 'shapely']
        COLAB_PROJECT_HINTS = [
            "/content/drive/MyDrive/MonolithFarm",
            "/content/drive/My Drive/MonolithFarm",
            "/content/Projeto-FarmLab",
            "/content/MonolithFarm",
        ]
        COLAB_DATA_HINTS = [f"{project_dir}/data" for project_dir in COLAB_PROJECT_HINTS]


        def package_available(name: str) -> bool:
            return importlib.util.find_spec(name) is not None


        def first_existing_path(candidates: list[str]) -> Path | None:
            for candidate in candidates:
                path = Path(candidate).expanduser()
                if path.exists():
                    return path.resolve()
            return None


        def discover_paths_config_file() -> Path | None:
            candidates: list[Path] = []
            if CONFIG_ENV_PATH:
                candidates.append(Path(CONFIG_ENV_PATH).expanduser())

            current = Path.cwd().resolve()
            for candidate in [current, *current.parents]:
                candidates.append(candidate / ".monolithfarm.paths.json")
                candidates.append(candidate / "monolithfarm.paths.json")

            seen: set[str] = set()
            for candidate in candidates:
                resolved = candidate.resolve() if candidate.is_absolute() else candidate
                key = str(resolved)
                if key in seen:
                    continue
                seen.add(key)
                if resolved.exists():
                    return resolved
            return None


        def load_paths_config() -> tuple[dict, Path | None]:
            config_path = discover_paths_config_file()
            if config_path is None:
                return {}, None

            payload = json.loads(config_path.read_text(encoding="utf-8"))
            if not isinstance(payload, dict):
                raise ValueError(f"Arquivo de configuracao invalido: {config_path}")
            return payload, config_path


        PATHS_CONFIG, PATHS_CONFIG_FILE = load_paths_config()
        CONFIG_BASE_DIR = PATHS_CONFIG_FILE.parent.resolve() if PATHS_CONFIG_FILE is not None else None
        DEFAULT_PROFILE = str(PATHS_CONFIG.get("default_profile") or "").strip()
        ACTIVE_PROFILE = PROFILE_NAME or DEFAULT_PROFILE
        PROFILE_SETTINGS = {}
        if ACTIVE_PROFILE:
            profiles = PATHS_CONFIG.get("profiles", {})
            if ACTIVE_PROFILE not in profiles:
                raise KeyError(
                    f"Perfil `{ACTIVE_PROFILE}` nao encontrado em {PATHS_CONFIG_FILE}."
                )
            profile_value = profiles.get(ACTIVE_PROFILE)
            if isinstance(profile_value, dict):
                PROFILE_SETTINGS = profile_value
        elif isinstance(PATHS_CONFIG.get("profile"), dict):
            PROFILE_SETTINGS = PATHS_CONFIG["profile"]


        def config_value(key: str):
            return PROFILE_SETTINGS.get(key) if isinstance(PROFILE_SETTINGS, dict) else None


        def resolve_config_path(raw_value: object, *, base_dir: Path | None) -> Path | None:
            if raw_value in {None, ""}:
                return None
            path = Path(str(raw_value)).expanduser()
            if not path.is_absolute() and base_dir is not None:
                path = (base_dir / path).resolve()
            else:
                path = path.resolve()
            return path


        def mount_colab_drive_if_needed() -> None:
            if not USE_COLAB_MODE:
                return
            if not IN_COLAB_RUNTIME:
                raise RuntimeError(
                    "MONOLITHFARM_NOTEBOOK_MODE='colab' foi definido, mas o runtime atual nao e Google Colab."
                )
            from google.colab import drive

            drive_root = Path("/content/drive")
            drive_root.mkdir(parents=True, exist_ok=True)
            if not (drive_root / "MyDrive").exists() and not (drive_root / "My Drive").exists():
                drive.mount("/content/drive")


        def find_project_dir() -> Path:
            explicit = os.environ.get("MONOLITHFARM_PROJECT_DIR")
            if explicit:
                return Path(explicit).expanduser().resolve()

            config_project_dir = resolve_config_path(config_value("project_dir"), base_dir=CONFIG_BASE_DIR)
            if config_project_dir is not None:
                return config_project_dir

            if USE_COLAB_MODE:
                hinted_project = first_existing_path(COLAB_PROJECT_HINTS)
                if hinted_project is not None and (hinted_project / "pyproject.toml").exists():
                    return hinted_project
                hinted_data = first_existing_path(COLAB_DATA_HINTS)
                if hinted_data is not None and (hinted_data.parent / "pyproject.toml").exists():
                    return hinted_data.parent.resolve()

            current = Path.cwd().resolve()
            for candidate in [current, *current.parents]:
                if (candidate / "pyproject.toml").exists():
                    return candidate

            raise FileNotFoundError(
                "Nao foi possivel localizar `pyproject.toml`. Defina MONOLITHFARM_PROJECT_DIR, "
                "MONOLITHFARM_PROFILE ou crie `.monolithfarm.paths.json`."
            )


        def find_data_dir(project_dir: Path) -> Path:
            explicit = os.environ.get("MONOLITHFARM_DATA_DIR")
            if explicit:
                return Path(explicit).expanduser().resolve()

            config_data_dir = resolve_config_path(config_value("data_dir"), base_dir=CONFIG_BASE_DIR)
            if config_data_dir is not None:
                return config_data_dir

            if USE_COLAB_MODE:
                hinted_data = first_existing_path(COLAB_DATA_HINTS)
                if hinted_data is not None:
                    return hinted_data

            for candidate in [project_dir / "data", project_dir / "FarmLab"]:
                if candidate.exists():
                    return candidate.resolve()

            return (project_dir / "data").resolve()


        def find_output_dir(project_dir: Path) -> Path:
            explicit = os.environ.get("MONOLITHFARM_OUTPUT_DIR")
            if explicit:
                return Path(explicit).expanduser().resolve()

            config_output_dir = resolve_config_path(config_value("output_dir"), base_dir=CONFIG_BASE_DIR)
            if config_output_dir is not None:
                return config_output_dir

            config_output_root = resolve_config_path(config_value("output_root"), base_dir=CONFIG_BASE_DIR)
            if config_output_root is not None:
                return (config_output_root / OUTPUT_SUBDIR).resolve()

            return (project_dir / OUTPUT_SUBDIR).resolve()


        def auto_install_enabled() -> bool:
            explicit = os.environ.get("MONOLITHFARM_AUTO_INSTALL")
            if explicit is not None:
                return explicit == "1"

            config_auto_install = config_value("auto_install")
            if config_auto_install is not None:
                return bool(config_auto_install)

            return USE_COLAB_MODE


        mount_colab_drive_if_needed()
        PROJECT_DIR = find_project_dir()
        DATA_DIR = find_data_dir(PROJECT_DIR)
        OUTPUT_DIR = find_output_dir(PROJECT_DIR)
        AUTO_INSTALL = auto_install_enabled()

        if str(PROJECT_DIR) not in sys.path:
            sys.path.insert(0, str(PROJECT_DIR))

        if AUTO_INSTALL and any(not package_available(name) for name in REQUIRED_PACKAGES):
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
            except Exception:
                subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-e", str(PROJECT_DIR)])

        print("NOTEBOOK_MODE =", NOTEBOOK_MODE)
        print("IN_COLAB_RUNTIME =", IN_COLAB_RUNTIME)
        print("USE_COLAB_MODE =", USE_COLAB_MODE)
        print("PROFILE_NAME =", PROFILE_NAME or "<default>")
        print("PATHS_CONFIG_FILE =", PATHS_CONFIG_FILE)
        print("ACTIVE_PROFILE =", ACTIVE_PROFILE or "<none>")
        print("PROJECT_DIR =", PROJECT_DIR)
        print("DATA_DIR    =", DATA_DIR)
        print("OUTPUT_DIR  =", OUTPUT_DIR)
        print("AUTO_INSTALL =", AUTO_INSTALL)

        if not DATA_DIR.exists():
            raise FileNotFoundError(f"Diretorio de dados nao encontrado: {DATA_DIR}")


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML

from farmlab.ndvi_crispdm import build_ndvi_crispdm_workspace, save_ndvi_crispdm_outputs


workspace = build_ndvi_crispdm_workspace(DATA_DIR)
data_audit = workspace["data_audit"]
pair_effect_tests = workspace["pair_effect_tests"]
event_driver_lift = workspace["event_driver_lift"]
transition_model_frame = workspace["transition_model_frame"]
transition_model_summary = workspace["transition_model_summary"]
transition_model_coefficients = workspace["transition_model_coefficients"]
transition_model_predictions = workspace["transition_model_predictions"]
final_hypothesis_register = workspace["final_hypothesis_register"]
decision_summary = workspace["decision_summary"]
ndvi_phase_timeline = workspace["ndvi_phase_timeline"]
ndvi_events = workspace["ndvi_events"]
ndvi_clean = workspace["ndvi_clean"]

print("Workspace CRISP-DM carregado.")


In [ ]:
def sample_rows_per_area(frame: pd.DataFrame, max_images_per_area: int = 3) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    groups = []
    for _, group in frame.sort_values("date").groupby("season_id", sort=False):
        if len(group) <= max_images_per_area:
            groups.append(group)
            continue
        positions = sorted({0, len(group) // 2, len(group) - 1})
        groups.append(group.iloc[positions[:max_images_per_area]])
    return pd.concat(groups, ignore_index=True)


def _image_to_base64(path: str) -> str | None:
    file_path = Path(path)
    if not file_path.exists():
        return None
    return base64.b64encode(file_path.read_bytes()).decode("ascii")


def gallery_html(frame: pd.DataFrame, *, title: str, subtitle_columns: list[str]) -> HTML:
    if frame.empty:
        return HTML(f"<h2>{title}</h2><p>Sem imagens disponiveis.</p>")

    cards = [f"<h2>{title}</h2>"]
    for row in frame.itertuples(index=False):
        image_b64 = _image_to_base64(str(row.image_path)) if pd.notna(row.image_path) else None
        subtitle = " | ".join(f"{column}={getattr(row, column)}" for column in subtitle_columns if hasattr(row, column))
        cards.append(f"<h3>{row.area_label} | {pd.Timestamp(row.date).date()}</h3>")
        cards.append(f"<p>{subtitle}</p>")
        if image_b64 is None:
            cards.append("<p>Imagem nao encontrada.</p>")
            continue
        cards.append(f'<img src="data:image/jpeg;base64,{image_b64}" style="max-width: 540px; border-radius: 8px; margin-bottom: 16px;" />')
    return HTML("\n".join(cards))


def event_gallery_html(events: pd.DataFrame, ndvi_clean: pd.DataFrame, max_events: int = 10) -> HTML:
    if events.empty or ndvi_clean.empty:
        return HTML("<h2>Galeria de eventos</h2><p>Sem eventos ou imagens disponiveis.</p>")

    priority = {"queda_forte": 0, "queda": 1, "baixo_vigor": 2, "recuperacao": 3, "pico": 4}
    selected = events.copy()
    selected["priority"] = selected["event_type"].map(priority).fillna(9)
    selected = selected.sort_values(["priority", "week_start", "area_label"]).head(max_events)

    cards = ["<h2>Eventos NDVI mais relevantes</h2>"]
    for event in selected.itertuples(index=False):
        candidates = ndvi_clean[ndvi_clean["season_id"] == event.season_id].copy()
        if candidates.empty:
            continue
        candidates["date"] = pd.to_datetime(candidates["date"], errors="coerce")
        event_date = pd.to_datetime(event.week_start, errors="coerce")
        match = candidates.loc[(candidates["date"] - event_date).abs().idxmin()]
        image_b64 = _image_to_base64(str(match["image_path"])) if pd.notna(match["image_path"]) else None
        cards.append(f"<h3>{event.area_label} | {pd.Timestamp(event.week_start).date()} | {event.event_type}</h3>")
        cards.append(f"<p><strong>Drivers:</strong> {event.drivers_summary}</p>")
        cards.append(f"<p>{event.story_sentence}</p>")
        if image_b64 is not None:
            cards.append(f'<img src="data:image/jpeg;base64,{image_b64}" style="max-width: 560px; border-radius: 8px; margin-bottom: 18px;" />')
    return HTML("\n".join(cards))


## 1. Business Understanding

Perguntas centrais:

- o que aconteceu em cada area;
- quais eventos e riscos explicam as quedas e recuperacoes do NDVI;
- o que os dados atuais indicam sobre convencional x 4.0;
- o que ainda depende de colheita final, custo e validacao agronomica.


In [ ]:
decision_summary


## 2. Data Understanding

Auditoria da base: cobertura de NDVI, clima, MIIP e operacao.


In [ ]:
data_audit


In [ ]:
fig = px.bar(
    data_audit,
    x="area_label",
    y=["ndvi_valid_ratio", "weather_coverage_ratio", "miip_coverage_ratio"],
    barmode="group",
    title="Cobertura por area: NDVI valido, clima e MIIP",
)
fig.update_layout(yaxis_title="Razao de cobertura", legend_title="")
fig


## 3. Data Preparation

A base final de modelagem foi montada em nivel semanal para preservar comparabilidade temporal e interpretabilidade.


In [ ]:
transition_model_frame.head(20)


## 4. Exploratory Analysis

Primeiro, a trajetoria semanal do NDVI por par.


In [ ]:
fig = px.line(
    ndvi_phase_timeline.sort_values("week_start"),
    x="week_start",
    y="ndvi_mean_week",
    color="area_label",
    facet_row="comparison_pair",
    markers=True,
    title="NDVI semanal por area e por par",
)
fig.update_layout(height=850)
fig


In [ ]:
heatmap = (
    ndvi_phase_timeline.pivot_table(
        index="area_label",
        columns="week_start",
        values="risk_flag_count",
        aggfunc="max",
    )
    .sort_index()
    .fillna(0)
)

fig = go.Figure(
    data=go.Heatmap(
        z=heatmap.values,
        x=[pd.Timestamp(value).strftime("%Y-%m-%d") for value in heatmap.columns],
        y=heatmap.index.tolist(),
        colorscale="YlOrRd",
        colorbar={"title": "Riscos"},
    )
)
fig.update_layout(title="Mapa de calor de risco por semana e area", xaxis_title="Semana", yaxis_title="Area")
fig


## 5. Statistical Testing

Nesta secao o par 4.0 x convencional e tratado como serie pareada por semana. Isso e mais forte do que comparar apenas medias agregadas.


In [ ]:
pair_effect_tests


In [ ]:
frame = pair_effect_tests.copy()
frame["metric_with_pair"] = frame["comparison_pair"] + " | " + frame["metric_label"]

fig = go.Figure()
for row in frame.itertuples(index=False):
    color = "#0f766e" if row.winner == "favorece_4_0" else "#b91c1c" if row.winner == "favorece_convencional" else "#475569"
    fig.add_trace(
        go.Scatter(
            x=[row.advantage_4_0],
            y=[row.metric_with_pair],
            error_x={
                "type": "data",
                "symmetric": False,
                "array": [max(row.ci_high - row.advantage_4_0, 0)],
                "arrayminus": [max(row.advantage_4_0 - row.ci_low, 0)],
            },
            mode="markers",
            marker={"size": 10, "color": color},
            showlegend=False,
            hovertemplate=(
                "Par/Metrica=%{y}<br>"
                "Vantagem 4.0=%{x:.4f}<br>"
                f"p={row.p_value:.4f}<br>"
                f"Evidencia={row.evidence_level}<extra></extra>"
            ),
        )
    )
fig.add_vline(x=0, line_dash="dash", line_color="#94a3b8")
fig.update_layout(
    title="Testes pareados do NDVI e riscos por par",
    xaxis_title="Vantagem do 4.0 sobre o convencional",
    yaxis_title="",
    height=820,
)
fig


## 6. Driver Analysis

Frequencia relativa dos drivers nas semanas problema em relacao ao restante da safra.


In [ ]:
event_driver_lift


In [ ]:
fig = px.bar(
    event_driver_lift,
    x="driver",
    y="delta_pp",
    color="evidence_level",
    facet_row="comparison_pair",
    title="Drivers sobre-representados nas semanas problema do NDVI",
    color_discrete_map={"alta": "#0f766e", "media": "#c58b00", "baixa": "#64748b"},
)
fig.update_layout(height=760)
fig


In [ ]:
event_gallery_html(ndvi_events, ndvi_clean, max_events=12)


## 7. Modeling

O target modelado e o `delta do NDVI da proxima semana`.

Foi utilizado um modelo linear regularizado e interpretavel para identificar sinais que pressionam ou aliviam a trajetoria do NDVI.


In [ ]:
transition_model_summary


In [ ]:
transition_model_coefficients.head(20)


In [ ]:
fig = px.bar(
    transition_model_coefficients.head(20),
    x="coefficient",
    y="feature",
    orientation="h",
    color="direction",
    title="Coeficientes padronizados do modelo de transicao do NDVI",
    color_discrete_map={"aumenta_ndvi_futuro": "#0f766e", "pressiona_ndvi_futuro": "#b91c1c"},
)
fig.update_layout(height=760, yaxis={"categoryorder": "total ascending"})
fig


In [ ]:
fig = px.scatter(
    transition_model_predictions,
    x="target_next_ndvi_delta",
    y="loo_predicted_next_ndvi_delta",
    color="area_label",
    facet_row="comparison_pair",
    title="Ajuste leave-one-area-out: delta real vs previsto do NDVI",
    labels={
        "target_next_ndvi_delta": "Delta real da proxima semana",
        "loo_predicted_next_ndvi_delta": "Delta previsto",
    },
)
fig.add_shape(type="line", x0=-0.2, y0=-0.2, x1=0.2, y1=0.2, line={"dash": "dash", "color": "#94a3b8"})
fig.update_layout(height=760)
fig


## 8. Evaluation

Sintese das hipoteses suportadas, negadas e inconclusivas.


In [ ]:
final_hypothesis_register


In [ ]:
decision_summary


## 9. Sintese Final

Esta secao separa:

- o que esta provado com os dados atuais;
- o que e apenas hipotese plausivel;
- o que ainda depende de custo, colheita final ou validacao agronomica;
- quais proximos passos fecham a decisao de negocio.


In [ ]:
sample_rows = sample_rows_per_area(ndvi_clean, max_images_per_area=2)
gallery_html(
    sample_rows,
    title="Galeria visual de apoio por area",
    subtitle_columns=["comparison_pair", "treatment", "ndvi_mean", "soil_pct", "dense_veg_pct"],
)


In [ ]:
pd.DataFrame({"gap": workspace["deep_dive_gaps"]})


In [ ]:
written_files = save_ndvi_crispdm_outputs(workspace, OUTPUT_DIR)
pd.DataFrame({"written_file": [str(path) for path in written_files]})
